# Data Cleaning and Preparation

## Initial data exploration and cleaning

In [ ]:
import pandas as pd
import json

In [ ]:
with open('ICT_Bachelors_ENG.json') as f:
    data = json.load(f)

courses = [item['data']['realizationById'] for item in data]
courses_df = pd.json_normalize(courses)

courses_df.head()

### Getting some information on courses and their teachers

**This is important information, used in testing whether the junction dataframe I create later is correct**

Checking which courses are taught by a specific teacher:

In [ ]:
filtered_df = courses_df[courses_df['teachers'].apply(lambda x: any(d['name'] == 'Jouko Kotkansalo' for d in x) if isinstance(x, list) else False)]

print('Courses where Jouko Kotkansalo is the teacher: \n\n',filtered_df['title'])

Checking the teachers for a course that I know has more than one teacher:

In [ ]:
course_row = courses_df[courses_df['code'] == 'HTGP0500-3005']
teachers_list = course_row['teachers'].values[0]

for teacher in teachers_list:
    print(teacher['name'])

### Continuation of cleaning

In [ ]:
courses_df.dropna(how='all', inplace=True)
print(f'Courses after dropping empty rows: {courses_df.shape[0]}')

In [ ]:
courses_df.columns

Useful columns in the data:

- id
- code
- title
- credits
- __typename
- objective
- teachingMethods
- content
- unit.title
- learningMaterial
- teachers
- teachingMethod
- teachingMethodFull.online
- teachingMethodFull.contact
- qualifications
- evaluation
- employerConnections
- examSchedule
- internationalConnections
- workload
- contentScheduling
- informationOfCourse
- furtherInformation
- evaluationScale
- seats.min
- seats.max
- relatedOffering

In [ ]:
columns_to_keep = [
    'id', 'code', 'title', 'credits', '__typename', 'objective', 'teachingMethods', 
    'content', 'unit.title', 'learningMaterial', 'teachers', 'teachingMethod', 
    'teachingMethodFull.online', 'teachingMethodFull.contact', 'qualifications', 
    'evaluation', 'employerConnections', 'examSchedule', 'internationalConnections', 
    'workload', 'contentScheduling', 'informationOfCourse', 'furtherInformation', 
    'evaluationScale', 'seats.min', 'seats.max', 'relatedOffering'
]

# Drop other columns
columns_to_drop = [col for col in courses_df.columns if col not in columns_to_keep]
courses_df = courses_df.drop(columns=columns_to_drop)

In [ ]:
courses_df.rename(columns={'__typename': 'type_name', 'teachingMethods': 'teaching_methods', 'teachingMethod': 'teaching_method',
                            'learningMaterial': 'learning_material', 'unit.title': 'unit_title', 'teachingMethod': 'teaching_method',
                            'teachingMethodFull.online': 'teaching_method_online', 'teachingMethodFull.contact': 'teaching_method_contact',
                            'employerConnections': 'employer_connections', 'examSchedule': 'exam_schedule', 'internationalConnections': 'international_connections',
                            'contentScheduling': 'content_scheduling', 'informationOfCourse': 'course_information', 'furtherInformation': 'further_information',
                            'evaluationScale': 'evaluation_scale', 'seats.min': 'min_seats', 'seats.max': 'max_seats', 'relatedOffering': 'related_offering'}
                            , inplace=True)

Checking to see if `id` and `code` are always the same, if they are we can drop `id` and use `code` as the primary key:

In [ ]:
rows_where_unequal = len(courses_df[courses_df['id'] != courses_df['code']])
print(f'Number of rows where id and code are not equal: {rows_where_unequal}')

In [ ]:
courses_df.drop(columns=['id'], inplace=True)

Rename `code` column to `course_code`:

In [ ]:
courses_df.rename(columns={'code': 'course_code'}, inplace=True)

## Database design

Finding out which columns contain objects, and making them into their own dataframes, which will be separate tables in the database:

In [ ]:
object_cols = courses_df.columns[courses_df.map(lambda x: isinstance(x, (list, dict))).any()].tolist()

print(f'Object columns: {object_cols}')

In [ ]:
teachers_data = []
related_offering_data = []

for index, row in courses_df.iterrows():
    teachers = row['teachers']
    if teachers is not None:
        for teacher in teachers:
            teachers_data.append({'name': teacher['name'], 'type': teacher['__typename'], 'course_code': row['course_code']})

    related_offerings = row['related_offering']
    if related_offerings is not None:
        for related_offering in related_offerings:
            related_offering_data.append({'related_offering_code': related_offering['code'], 'title': related_offering['title'],
                                          'link': related_offering['link'], 'type': related_offering['__typename'],
                                          'course_code': row['course_code']})

teachers_df = pd.DataFrame(teachers_data)
related_offerings_df = pd.DataFrame(related_offering_data)

In [ ]:
teachers_df.head()

In [ ]:
related_offerings_df.head()

Dropping `teachers` and `related_offering` from the course data, as they now have their own dataframes.

In [ ]:
courses_df.drop(columns=['teachers', 'related_offering'], inplace=True)

In [ ]:
pd.set_option('display.max_columns', None)
courses_df.head()

Drop `evaluation` column, as it's empty:

In [ ]:
courses_df.drop(columns=['evaluation'], inplace=True)

Finding out whether some courses have multiple teachers:

In [ ]:
rows_where_unequal = len(teachers_df[teachers_df.duplicated(subset=['course_code'], keep=False)])
print(f'Number of rows where different teachers have the same course code: {rows_where_unequal}')

# Examples
teachers_df[teachers_df.duplicated(subset=['course_code'], keep=False)].head(10)

Finding out whether some teachers teach multiple courses:

In [ ]:
duplicate_rows = teachers_df[teachers_df.duplicated('name', keep=False)]

# Print some examples
print(duplicate_rows.sort_values('name').head())

Creating a new dataframe storing the relationships between courses and teachers, which will be a junction table in the database:

In [ ]:
# Create a unique teachers_df DataFrame where each teacher has a unique ID
unique_teachers_df = teachers_df[['name']].drop_duplicates()
unique_teachers_df['id'] = range(1, len(unique_teachers_df) + 1)

# Create a junction table by mapping the teacher names in the original DataFrame to their corresponding IDs
junction_df = teachers_df[['name', 'course_code']].copy()
junction_df = junction_df.merge(unique_teachers_df, on='name')
junction_df = junction_df[['id', 'course_code']]
junction_df.columns = ['teacher_id', 'course_code']

print(junction_df.head())

In [ ]:
unique_teachers_df.head()

In [ ]:
junction_df[junction_df['teacher_id'] == 2]

In [ ]:
teachers_df[teachers_df['name'] == 'Jouko Kotkansalo']

In [ ]:
courses_df[courses_df['course_code'].isin(['TTC2040-3019', 'TTC8850-3004', 'TTC1060-3029', 'TTC8850-3003'])]

These are the same courses we discovered earlier as being taught by this teacher, so the junction table is correct.

In [ ]:
junction_df[junction_df['course_code'] == 'HTGP0500-3005']

Renaming `id` column in `unique_teachers_df` to `teacher_id` and inspecting the names for this course:

In [ ]:
unique_teachers_df.rename(columns={'id': 'teacher_id'}, inplace=True)

In [ ]:
unique_teachers_df[unique_teachers_df['teacher_id'].isin([4, 5])]

These also line up with the names we found earlier!

## Final inspection of dataframes, and saving to .csv

In [ ]:
courses_df.head()

In [ ]:
unique_teachers_df.head()

In [ ]:
junction_df.head()

**I won't be using related offerings in my application at the moment, so I'll leave this out. I want to create my own recommendations in a later feature.**

In [ ]:
related_offerings_df.head()

Modifying datatypes to match the database:

In [ ]:
courses_df.dtypes

In [ ]:
courses_df = courses_df.astype({col: 'string' for col in courses_df.columns if col not in ['credits', 'min_seats', 'max_seats']})
courses_df = courses_df.astype({'credits': 'float', 'min_seats': 'float', 'max_seats': 'float', 'teaching_method_online': 'float', 'teaching_method_contact': 'float'})

courses_df.dtypes

In [ ]:
unique_teachers_df.dtypes

In [ ]:
unique_teachers_df = unique_teachers_df.astype({'name': 'string'})
unique_teachers_df.dtypes

In [ ]:
junction_df.dtypes

In [ ]:
junction_df = junction_df.astype({'course_code': 'string'})
junction_df.dtypes

In [ ]:
unique_teachers_df = unique_teachers_df[['teacher_id', 'name']]
unique_teachers_df.head()

Dropping `teacher_id` from the teachers dataframe, as the database will generate this automatically:

In [ ]:
unique_teachers_df.drop(columns=['teacher_id'], inplace=True)

Finally, saving the dataframes to .csv files:

In [ ]:
courses_df.to_csv('courses.csv', index=False, escapechar='\\')
unique_teachers_df.to_csv('teachers.csv', index=False, escapechar='\\')
junction_df.to_csv('junction.csv', index=False, escapechar='\\')